# Step 1 — CTGAN/DPGAN n_iter convergence + timing check (Colab T4)

Runs `benchmark_tapas/neural_tuning/convergence_check.py`: trains each GAN once on the FULL training set at n_iter ∈ {50, 100, 200, default}, recording GPU fit-time and TSTR utility (`performance.xgb.syn_id`). Use the output to pick the smallest n_iter where utility has plateaued, and to check whether the neural counts fit one ~2 hr T4 session.

**Upload to `MyDrive/VRI/experimentation/` first:** `benchmark_tapas/config.py`, `benchmark_tapas/neural_tuning/convergence_check.py`, `data/adult_train.csv` (and `adult_test.csv`).

Writes incrementally to `results/convergence/convergence_check.csv` on Drive and is **resumable** (already-done (method, n_iter) rows are skipped).

> Sweep cost on a T4 ≈ 30–45 min for both GANs — it may eat most of one day's session, so run the real neural audits on a later session.

## 1. Install pinned deps (restarts runtime — expected)

In [ ]:
# TAPAS's pyproject.toml declares python ">=3.9,<3.11" (poetry-core rejects
# Colab's py3.12) and pandas ^1.4.1 (no py3.12 wheel). Clone the pinned commit,
# patch the python constraint, install --no-deps, then pin the rest explicitly
# (same recipe as the run_*_colab notebooks). convergence_check.py itself only
# needs synthcity + torch, but we keep the identical env so timings match the
# real neural runs.
!rm -rf /content/tapas_src
!git clone -q https://github.com/alan-turing-institute/tapas.git /content/tapas_src
!cd /content/tapas_src && git checkout -q a7069d7e040828db0da174d1b003fa03a98e5453
!sed -i 's/python = ">=3.9, <3.11"/python = ">=3.9"/' /content/tapas_src/pyproject.toml
!pip install /content/tapas_src --no-deps -q
!pip install palettable==3.3.3 -q
!pip install synthcity==0.2.12 -q
!pip install opacus==1.4.1 -q
!pip install pandas==2.3.3 -q

import os
os.kill(os.getpid(), 9)  # force restart to clear numpy/torch binary conflicts

## 2. Verify GPU (stop here if CUDA is False)

In [ ]:
import torch
print('torch:', torch.__version__, '| CUDA available:', torch.cuda.is_available())
if not torch.cuda.is_available():
    print('\n*** No CUDA GPU. Runtime > Change runtime type > T4 GPU, then re-run. ***')
    print('convergence_check.py auto-detects the device; timings are only')
    print('meaningful (and the sweep only tractable) on the GPU.')
else:
    print('GPU OK.')
!nvidia-smi

## 3. Mount Drive + symlink benchmark_tapas/

In [ ]:
from google.colab import drive
import os
drive.mount('/content/drive')
DRIVE_BASE = '/content/drive/MyDrive/VRI/experimentation'
os.chdir('/content')

# Symlink benchmark_tapas/ from Drive so results/convergence/convergence_check.csv (and its
# incremental, resumable rows) write straight onto Drive -- a disconnected
# session RESUMES: already-recorded (method, n_iter) rows are skipped.
if not os.path.exists('/content/benchmark_tapas'):
    os.symlink(f'{DRIVE_BASE}/benchmark_tapas', '/content/benchmark_tapas')

required = [
    f'{DRIVE_BASE}/benchmark_tapas/config.py',
    f'{DRIVE_BASE}/benchmark_tapas/neural_tuning/convergence_check.py',
    f'{DRIVE_BASE}/data/adult_train.csv',
]
missing = [p for p in required if not os.path.exists(p)]
if missing:
    print('*** MISSING on Drive -- upload before running: ***')
    for p in missing: print('  ', p)
else:
    print('All required files present. Symlink ready.')

## 4. Run the sweep (safe to re-run after a disconnect — it resumes)

In [ ]:
!python /content/benchmark_tapas/neural_tuning/convergence_check.py